#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim,col

In [0]:
RENAME_MAP ={
    'sls_ord_num':'order_number',
    'sls_prd_key':'product_key',
    'sls_cust_id':'customer_id',
    'sls_order_dt':'order_date',
    'sls_ship_dt':'ship_date',
    'sls_due_dt':'due_date',
    'sls_sales':'sales_amount',
    'sls_quantity':'quantity',
    'sls_price':'price'
}

#Reading Data from Bronze Table

In [0]:
df = spark.table("workspace.bronze.crm_sales_info")

#Same Data Transformations

In [0]:
df = (
      df.withColumn("sls_ord_num",F.col("sls_ord_num").cast("string"))
      .withColumn("sls_prd_key",F.col("sls_prd_key").cast("string"))
      .withColumn("sls_cust_id",F.col("sls_cust_id").cast("int"))
      .withColumn("sls_order_dt",F.try_to_date(F.col("sls_order_dt").cast("string"),"yyyyMMdd"))
      .withColumn("sls_ship_dt",F.try_to_date(F.col("sls_ship_dt").cast("string"),"yyyyMMdd"))
      .withColumn("sls_due_dt",F.try_to_date(F.col("sls_due_dt").cast("string"),"yyyyMMdd"))
      .withColumn("sls_sales",F.col("sls_sales").cast("decimal(12,2)"))
      .withColumn("sls_quantity",F.col("sls_quantity").cast("int"))
      .withColumn("sls_price",F.col("sls_price").cast("decimal(12,2)"))
)


In [0]:
df = (df.withColumn("is_return",F.col("sls_price") < 0)
      .withColumn(
          "processing_days",F.datediff(F.col("sls_ship_dt"),F.col("sls_order_dt")))
      .withColumn(
          "expected_delivery_days",F.datediff(F.col("sls_due_dt"),F.col("sls_ship_dt")))
      .withColumn("order_year",F.year(F.col("sls_order_dt")))
      .withColumn("order_month",F.month(F.col("sls_order_dt")))
    )

In [0]:
df = df.filter(F.col("sls_ord_num").isNotNull())
df = df.filter(F.col("sls_cust_id").isNotNull())
df = df.filter(F.col("sls_prd_key").isNotNull())

df = df.dropDuplicates()

In [0]:
df.filter(F.col("sls_ord_num").isNull()).count()
df.filter(F.col("sls_cust_id").isNull()).count()
df.filter(F.col("sls_prd_key").isNull()).count()

In [0]:
df.filter(F.col("sls_ship_dt") < F.col("sls_order_dt")).count()
df.filter(F.col("sls_due_dt") < F.col("sls_ship_dt")).count()


In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType,StringType):
        df = df.withColumn(field.name,trim(col(field.name)))
    

#Renaming the Columns

In [0]:
for old_name , new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name,new_name)

In [0]:
df.display()

In [0]:
df.filter(F.col("sales_amount") != F.col("price") * F.col("quantity")).show()
df.filter(F.col("sales_amount") != F.col("price") * F.col("quantity")).count()
df.filter(F.col("is_return") == ("true")).count()


#Write into Silver Table

In [0]:
(
    df.write
        .mode("overwrite")
        .option("overwriteSchema","true")
        .format("delta")
        .saveAsTable("silver.crm_sales_info")
)

In [0]:
df.count()

In [0]:

df.select(
    "order_number",
    "product_key",
    "customer_id",
    "quantity",
    "sales_amount",
    "price"
).show(20, truncate=False)

In [0]:
# إجمالي الصفوف
print("Total rows:", df.count())

# عدد الطلبات المختلفة
print(
    "Distinct orders:",
    df.select("order_number").distinct().count()
)

# الطلبات التي تحتوي على أكثر من صف
df.groupBy("order_number") \
    .count() \
    .filter(F.col("count") > 1) \
    .orderBy(F.col("count").desc()) \
    .show(20, truncate=False)

In [0]:
# افحص تفاصيل طلب متكرر
df.filter(
    F.col("order_number") == "SO72656"
).select(
    "order_number",
    "product_key",
    "customer_id",
    "quantity",
    "sales_amount",
    "price"
).show(20, truncate=False)

In [0]:
# هل نفس المنتج بيتكرر داخل نفس الطلب؟
df.groupBy(
    "order_number",
    "product_key"
).count().filter(
    F.col("count") > 1
).orderBy(
    F.col("count").desc()
).show(20, truncate=False)